# Retail Customer Purchase & Churn Intelligence Engine
### Problem Statement 2: AI/ML-Powered Predictive Insight Dashboard

This notebook provides an end-to-end, memory-optimized training and artifact export pipeline:
1. **Chunked Ingestion & Cleaning**: Ingests multi-GB transaction logs safely without exceeding Kaggle's 13GB RAM.
2. **Leak-Free RFM Engineering**: Builds behavioral customer features using a strict 75% timeline cutoff.
3. **Model Training**: Fits a balanced Random Forest Classifier.
4. **Holdout Evaluation**: Evaluates ROC-AUC, F1, Accuracy, Precision, and Recall on unseen test data.
5. **Fixed Cutoff Calibration**: Derives fixed High (80th pct) and Medium (50th pct) intent cutoffs.
6. **Artifact Export**: Exports all model files, contracts, and test data for the Streamlit dashboard.

In [ ]:
# ── Step 1: Discover Dataset & Set Optimized Dtypes ──
import os
import glob
import pandas as pd
import numpy as np

# Automatically find the retail CSV regardless of Kaggle input folder structure
possible_files = glob.glob('/kaggle/input/**/online_retail_II.csv', recursive=True) + \
                 glob.glob('/kaggle/input/**/online_retail*.csv', recursive=True)

if possible_files:
    data_file = possible_files[0]
    print(f"Found dataset at: {data_file}")
else:
    data_file = '/kaggle/input/datasets/mashlyn/online-retail-ii-uci/online_retail_II.csv'
    print(f"Using default path: {data_file}")

# Compact 32-bit types to prevent RAM crashes on multi-GB files
dtype_spec = {
    'Invoice': 'str', 'InvoiceNo': 'str', 'StockCode': 'str', 'Description': 'str',
    'Quantity': 'float32', 'Price': 'float32', 'UnitPrice': 'float32',
    'Customer ID': 'float32', 'CustomerID': 'float32', 'Country': 'str'
}

In [ ]:
# ── Step 2: Chunked Ingestion & Cleaning (RAM-Safe for Multi-GBs) ──
print("Starting chunked stream processing...")
chunks = []
total_raw_rows = 0
chunk_size = 150_000

rename_map = {'InvoiceNo': 'Invoice', 'UnitPrice': 'Price', 'CustomerID': 'Customer ID'}

for chunk in pd.read_csv(data_file, encoding='ISO-8859-1', chunksize=chunk_size, dtype=dtype_spec, low_memory=False):
    total_raw_rows += len(chunk)
    chunk = chunk.rename(columns={k: v for k, v in rename_map.items() if k in chunk.columns})
    
    # In-flight filter: drop missing Customer IDs, negative/zero quantities, and cancellations
    chunk = chunk.dropna(subset=['Customer ID'])
    chunk = chunk[~chunk['Invoice'].astype(str).str.startswith('C')]
    chunk = chunk[(chunk['Quantity'] > 0) & (chunk['Price'] > 0)]
    
    # Type downcasting
    chunk['Customer ID'] = chunk['Customer ID'].astype(np.int32)
    chunk['InvoiceDate'] = pd.to_datetime(chunk['InvoiceDate'])
    chunk['LineTotal'] = (chunk['Quantity'] * chunk['Price']).astype(np.float32)
    
    clean_cols = ['Customer ID', 'Invoice', 'StockCode', 'InvoiceDate', 'LineTotal', 'Country']
    chunks.append(chunk[clean_cols])

df_clean = pd.concat(chunks, ignore_index=True)
del chunks

print(f"Ingestion complete!")
print(f"Raw rows read:       {total_raw_rows:,}")
print(f"Cleaned rows kept:   {df_clean.shape[0]:,}")
print(f"Unique customers:    {df_clean['Customer ID'].nunique():,}")
print(f"Full date range:     {df_clean['InvoiceDate'].min()} -> {df_clean['InvoiceDate'].max()}")

In [ ]:
# ── Step 3: Time-Based Split (Strict Leakage Prevention) ──
full_start = df_clean['InvoiceDate'].min()
full_end = df_clean['InvoiceDate'].max()
total_days = (full_end - full_start).days

# 75% cutoff: features built strictly from first 75%, churn labeled from final 25%
cutoff_date = full_start + pd.Timedelta(days=int(total_days * 0.75))
print(f"Timeline: {total_days} total days")
print(f"Cutoff Date (75% mark): {cutoff_date}")
print(f"Feature Window:  {full_start} -> {cutoff_date}")
print(f"Label Window:    {cutoff_date} -> {full_end}")

feature_window = df_clean[df_clean['InvoiceDate'] <= cutoff_date]
label_window = df_clean[df_clean['InvoiceDate'] > cutoff_date]

print(f"Transactions in Feature Window: {feature_window.shape[0]:,}")
print(f"Transactions in Label Window:   {label_window.shape[0]:,}")

In [ ]:
# ── Step 4: RFM Feature Engineering from Feature Window ──
snapshot_date = cutoff_date + pd.Timedelta(days=1)

rfm = feature_window.groupby('Customer ID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('Invoice', 'nunique'),
    Monetary=('LineTotal', 'sum'),
    DistinctProducts=('StockCode', 'nunique'),
    Tenure=('InvoiceDate', lambda x: (snapshot_date - x.min()).days),
    TopCountry=('Country', lambda x: x.mode().iloc[0] if not x.mode().empty else 'Unknown')
).reset_index()

print(f"Engineered RFM profiles for {rfm.shape[0]:,} active customers.")
rfm.describe().round(2)

In [ ]:
# ── Step 5: Engineer Ground-Truth Churn Label ──
# Churned = Did not return to make a purchase in the post-cutoff label window
returning_customers = set(label_window['Customer ID'].unique())
rfm['Churned'] = (~rfm['Customer ID'].isin(returning_customers)).astype(int)

churn_rate = rfm['Churned'].mean()
print(f"Overall Customer Churn Rate: {churn_rate:.2%}")
print(rfm['Churned'].value_counts().rename({0: 'Retained (0)', 1: 'Churned (1)'}))

In [ ]:
# ── Step 6: Log Transformation & Categorical Bucketing ──
# Log1p transform tames severe monetary and frequency skewness without losing variance
for col in ['Monetary', 'DistinctProducts', 'Frequency']:
    rfm[f'{col}_log'] = np.log1p(np.maximum(0, rfm[col]))

# Bucket rare countries with fewer than 15 customers to prevent high dimensionality
country_counts = rfm['TopCountry'].value_counts()
common_countries = country_counts[country_counts >= 15].index.tolist()
rfm['CountryBucket'] = rfm['TopCountry'].apply(lambda x: x if x in common_countries else 'Other')

print("Country distribution after bucketing:")
print(rfm['CountryBucket'].value_counts())

In [ ]:
# ── Step 7: Stratified Train / Test Split & Preprocessing ──
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_features = ['Recency', 'Frequency_log', 'Monetary_log', 'DistinctProducts_log', 'Tenure']
categorical_features = ['CountryBucket']

X = rfm[numeric_features + categorical_features].copy()
y = rfm['Churned'].astype(int)

# Stratified 80/20 split preserves exact churn ratio in train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(f"Train size: {X_train.shape[0]:,} customers (Churn: {y_train.mean():.1%})")
print(f"Test size:  {X_test.shape[0]:,} customers (Churn: {y_test.mean():.1%})")

# Fit preprocessors strictly on training data
scaler = StandardScaler().fit(X_train[numeric_features])
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False).fit(X_train[categorical_features])

# Transform both splits
X_train_proc = np.hstack([
    scaler.transform(X_train[numeric_features]),
    encoder.transform(X_train[categorical_features])
])
X_test_proc = np.hstack([
    scaler.transform(X_test[numeric_features]),
    encoder.transform(X_test[categorical_features])
])

print(f"Processed matrix width: {X_train_proc.shape[1]} features")

In [ ]:
# ── Step 8: Train Balanced Random Forest Classifier ──
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=150,
    max_depth=10,
    min_samples_split=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

print("Training Random Forest classifier...")
model.fit(X_train_proc, y_train)
print("Model training complete!")

In [ ]:
# ── Step 9: Evaluate on Unseen Holdout Test Set ──
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report
)

test_probs = model.predict_proba(X_test_proc)[:, 1]
test_preds = model.predict(X_test_proc)

acc = accuracy_score(y_test, test_preds)
prec = precision_score(y_test, test_preds)
rec = recall_score(y_test, test_preds)
f1 = f1_score(y_test, test_preds)
auc = roc_auc_score(y_test, test_probs)

print("=" * 45)
print("             HOLDOUT TEST METRICS            ")
print("=" * 45)
print(f"  ROC-AUC:   {auc:.4f}  (Discrimination Power)")
print(f"  F1-Score:  {f1:.4f}  (Harmonic Precision/Recall)")
print(f"  Precision: {prec:.4f}  (Share of True Churns in Flags)")
print(f"  Recall:    {rec:.4f}  (Share of Churned Caught)")
print(f"  Accuracy:  {acc:.4f}")
print("=" * 45)

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, test_preds)
print(pd.DataFrame(cm, index=['Actual Retained (0)', 'Actual Churned (1)'],
                      columns=['Pred Retained (0)', 'Pred Churned (1)']))

print("\nDetailed Classification Report:")
print(classification_report(y_test, test_preds, target_names=['Retained (0)', 'Churned (1)']))

In [ ]:
# ── Step 10: Compute Fixed Calibrated Intent Cutoffs ──
# 80th percentile for High Intent, 50th percentile for Medium Intent
high_cutoff = float(np.percentile(test_probs, 80))
medium_cutoff = float(np.percentile(test_probs, 50))

print(f"High Intent Cutoff (80th pct):   {high_cutoff:.4f}")
print(f"Medium Intent Cutoff (50th pct): {medium_cutoff:.4f}")

# Verify distribution across tiers
test_tiers = pd.Series(test_probs).apply(
    lambda p: 'High Intent' if p >= high_cutoff else ('Medium Intent' if p >= medium_cutoff else 'Low Intent')
)
print("\nHoldout Distribution across Intent Tiers:")
print(test_tiers.value_counts(normalize=True).apply(lambda x: f"{x:.1%}"))

In [ ]:
# ── Step 11: Extract Feature Importance Rankings ──
cat_encoded_names = list(encoder.get_feature_names_out(categorical_features))
all_feature_names = numeric_features + cat_encoded_names

feature_importances_dict = {
    feat: round(float(imp), 5)
    for feat, imp in zip(all_feature_names, model.feature_importances_)
}

feature_importances_sorted = dict(
    sorted(feature_importances_dict.items(), key=lambda item: item[1], reverse=True)
)

print(f"Total features in model: {len(all_feature_names)}")
print("\nTop 7 Feature Drivers:")
for feat, imp in list(feature_importances_sorted.items())[:7]:
    print(f"  {feat:30s} -> {imp:.4f}")

In [ ]:
# ── Step 12: Build Contract Payloads (schema.json & metrics.json) ──
import json

schema = {
    "target": "Churned",
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "probability_class": 1,
    "model_type": "RandomForestClassifier",
    "probability_meaning": "Probability of Customer Churn (1 = Churned, 0 = Retained)"
}

metrics = {
    "accuracy": round(float(acc), 4),
    "precision": round(float(prec), 4),
    "recall": round(float(rec), 4),
    "f1": round(float(f1), 4),
    "roc_auc": round(float(auc), 4),
    "high_intent_cutoff": round(float(high_cutoff), 4),
    "medium_intent_cutoff": round(float(medium_cutoff), 4),
    "feature_importances": feature_importances_sorted
}

print("Contract JSON payloads successfully prepared!")
print(f"Holdout ROC-AUC: {metrics['roc_auc']} | F1: {metrics['f1']}")
print(f"Cutoffs -> High: {metrics['high_intent_cutoff']} | Medium: {metrics['medium_intent_cutoff']}")

In [ ]:
# ── Step 13: Export Model Artifacts to /kaggle/working/artifacts ──
import os
import joblib

artifacts_dir = '/kaggle/working/artifacts'
os.makedirs(artifacts_dir, exist_ok=True)

# Save estimators
joblib.dump(model, os.path.join(artifacts_dir, 'model.pkl'))
joblib.dump(encoder, os.path.join(artifacts_dir, 'encoder.pkl'))
joblib.dump(scaler, os.path.join(artifacts_dir, 'scaler.pkl'))

# Save contracts
with open(os.path.join(artifacts_dir, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)

with open(os.path.join(artifacts_dir, 'schema.json'), 'w') as f:
    json.dump(schema, f, indent=2)

print("Exported artifacts in /kaggle/working/artifacts:")
for fname in sorted(os.listdir(artifacts_dir)):
    fsize = os.path.getsize(os.path.join(artifacts_dir, fname))
    print(f"  {fname:20s} ({fsize:,} bytes)")

In [ ]:
# ── Step 14: Save 100-Row Sample Customer Batch for Streamlit App ──
data_dir = '/kaggle/working/data'
os.makedirs(data_dir, exist_ok=True)

sample_batch = X_test.head(100).copy()
sample_file = os.path.join(data_dir, 'sample_sessions.csv')
sample_batch.to_csv(sample_file, index=False)

print(f"Saved {sample_batch.shape[0]} test rows to: {sample_file}")
sample_batch.head(3)

In [ ]:
# ── Step 15: Create 1-Click Downloadable ZIP ──
import shutil

zip_filename = '/kaggle/working/retail_intelligence_artifacts'
shutil.make_archive(zip_filename, 'zip', '/kaggle/working')

print(f"Package ready: {zip_filename}.zip")
print("Download this file from the Kaggle Output panel on the right sidebar!")